### Ingesting Constructors File

1) I'll reading file using spark
2) Adding metadata columns or audit columns such as:

    -- Source File

    --Ingestion Timestamp

3) Once standandardized, I will be writting them to Bronze Table    

Reading the CSV file

In [0]:
%run "../00-Common/01. Env Config"

In [0]:
%run "../00-Common/02. Bronze-helper_func"

In [0]:
# Defining relevant Source and Table name

Source_file=f"{landing_folder_path}/constructors.json"
Table_name=f"{catalog_name}.{bronze_schema}.constructors"

Using DDL style formatting instead of Structfile. (Either can be used but for the sake of exploring & flexibility, DDL is beiing implemented)

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType, DateType


constructors_Schema="""constructorId STRING
                        , name STRING
                        , nationality STRING
                        , url STRING"""

constructors_Schema


In [0]:
races_df=(
                spark.read.format('json')
                #.option('header',True)
                #.option('inferSchema','true')
                .option('mode','FAILFAST')      # PERMISSIVE, DROPMALFORMED, FAILFAST
                .schema(constructors_Schema)
                .load(Source_file))

In [0]:
display(races_df)

2) Adding Metadata

In [0]:
Constructors_final_df= add_ingestion_metadata(races_df)

In [0]:
display(Constructors_final_df)

3) Witting data to Bronze layer/delta table

In [0]:
(
    Constructors_final_df
    .write
    .format('delta')
    .mode('overwrite')
    .saveAsTable(Table_name)
)

In [0]:
display(spark.table(Table_name))